## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on P3's modeling table, chooses the metrics, the hyperparameters and the decision threshold **without touching the test set**, evaluates the final choice once on the test set, explains the final model and lists its limitations.

**Inputs from Part 3:** the modeling table `model` (one row per driver per race, 2006-2024, with the `split` column), the features `FEATURES` and their groups `FEATURE_GROUPS`.
**Outputs for P1's inference function:** `FINAL_MODEL` (preprocessing + model in one pipeline; it takes the columns in `FEATURES`), `FINAL_THRESHOLD` and `FINAL_NAME`.

Sections: 4.1) evaluation function `train_eval()`, 4.2) data, preprocessing and leakage checks, 4.3) sanity check of the metrics, 4.4) baselines, 4.5) statistical models (plus two other model families), 4.6) shallow neural network (plus a wider and a deeper network), 4.7) hyperparameter tuning (plus the number of trees), 4.8) metric choice and threshold, 4.9) final model, 4.10) test set evaluation, 4.11) explainability (XAI), 4.12) model limitations, 4.13) hand-off to the inference function.

Every attempt has its own cell and its own result line, numbered **Attempt 1, 2, 3...** in the order they were run.

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
# In main.ipynb, Part 3 runs just before this part and defines `model`, FEATURES and FEATURE_GROUPS.
# Locally we get exactly the same objects by running P3's notebook first.
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED
from IPython.utils.capture import capture_output
with capture_output():          # hide Part 3's printouts and plots
    %run p3_features.ipynb

In [ ]:
clean = load_clean()
outcomes = clean["outcomes"]    # post-race columns (status, finishing position...). Never features: only used to tell the stories in 4.11-4.12
tables = clean["tables"]        # cleaned tables (driver and race names for the examples)

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (our models and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*, the standard summary of the precision-recall curve. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a probability threshold. The threshold is 0.5 until it is tuned on validation in 4.8.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it for the ablation.

In [ ]:
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "PR-AUC": average_precision_score(y, proba),
        "F1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model"):
    """Fit `model` on the training years only, then score it on train and validation.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    model.fit(X_train, y_train)
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Data, preprocessing and leakage checks

**Data:** P3's modeling table and P3's time split (train 2006-2019, validation 2020-2021, test 2022-2024). The test rows are separated only in 4.10.

**Features:** P3's `FEATURES` (20 numeric columns in 6 groups), **except `era_ordinal`**. The training years contain only the eras 0-2 (early V8, late V8, hybrid), the validation years only era 2, and the test years only era 3 (ground effect). A model would have to extrapolate a weight learned on 0-2 to a value it has never seen, and no validation row can check that. P3's own note in 3.7 says the same. Attempt 3 (4.4) shows the validation evidence. So the models use **19 features**. `MODEL_FEATURES` is derived from `FEATURES` automatically, so if P3 changes her list, this part follows.

**Preprocessing (one pipeline, the same for every model):**
* **Column selection:** the first step keeps the columns in `MODEL_FEATURES` and turns them into plain numbers (P3 stores some as nullable integers). The final model therefore still accepts a record with all of P3's `FEATURES` (P1's inference function passes exactly that) and ignores `era_ordinal`.
* **Missing values are structural, not errors:** the championship position is empty in round 1 of every season (no race before it), and qualifying times are missing for about 1% of rows (no timed lap). P3 left them as `NaN` on purpose. We fill them with the **training median** and add a **missing-indicator column** for every feature that has gaps in training, so the model still knows that the value was missing (for example "first race of the season").
* **Scaling:** every feature is standardized (mean 0, standard deviation 1). Logistic regression, k-NN and the network need it; for the tree models it changes nothing, so all models can share one pipeline.
* **Fitted on the training rows only:** the imputer and the scaler are steps of a `Pipeline`, so `fit()` learns the medians, means and standard deviations from the training years only; validation and test rows are only transformed.
* **Difference from P3's experiments:** P3's preprocessing experiments and ablation use plain median imputation without missing-value flags. Our models add the flags (`add_indicator=True`), so a missing value stays visible to the model. This is why P3's logistic-regression baseline and ours (4.4) differ slightly.

**Leakage checks (code below):** no post-race column is a feature, and each split holds only its own years.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

table = model                     # P3's modeling table (read only; the name `model` is kept for Part 3)

# leakage checks: no post-race column is a feature, and the time split is respected
assert not set(FEATURES) & set(outcomes.columns), "a post-race column is used as a feature"
assert table.loc[table["split"] == "train", "year"].max() <= 2019
assert table.loc[table["split"] == "validation", "year"].between(2020, 2021).all()
assert table.loc[table["split"] == "test", "year"].min() >= 2022

# the features the models use: P3's FEATURES without era_ordinal (reason in the text above)
NOT_USED = ["era_ordinal"]
MODEL_FEATURES = [f for f in FEATURES if f not in NOT_USED]
MODEL_GROUPS = {g: [f for f in cols if f in MODEL_FEATURES] for g, cols in FEATURE_GROUPS.items()}

train = table[table["split"] == "train"]
val = table[table["split"] == "validation"]
X_train, y_train = train[FEATURES].astype(float), train["podium"]     # all of P3's columns; the pipeline selects
X_val, y_val = val[FEATURES].astype(float), val["podium"]

def keep_model_features(X):
    """First step of every model: keep the MODEL_FEATURES columns (drops era_ordinal) as plain float numbers."""
    return X[MODEL_FEATURES].astype(float)

select = FunctionTransformer(keep_model_features, feature_names_out=lambda transformer, names: MODEL_FEATURES)
preprocess = make_pipeline(select, SimpleImputer(strategy="median", add_indicator=True), StandardScaler())


def with_preprocessing(estimator):
    """The shared preprocessing followed by `estimator`, as one pipeline (fitted on the training rows only)."""
    return Pipeline([("prep", clone(preprocess)), ("model", estimator)])


for name, part in [("train", train), ("validation", val)]:
    print(f"{name}: {len(part)} rows ({part['year'].min()}-{part['year'].max()}), "
          f"{part['podium'].sum()} podiums, podium rate {part['podium'].mean():.3f}")
print(f"P3's FEATURES: {len(FEATURES)}; used by the models: {len(MODEL_FEATURES)} (not used: {NOT_USED})")
print("era_ordinal values per split:", table.groupby("split")["era_ordinal"].unique().apply(sorted).to_dict())
display(X_train[MODEL_FEATURES].isna().mean().loc[lambda s: s > 0].round(4).to_frame("share missing (train)"))

**What we see.**
* **Rows:** 5,742 training rows (2006-2019, 804 podiums, 14.0%) and 774 validation rows (2020-2021, 117 podiums, 15.1%). The two podium rates are close, so the splits are comparable.
* **`era_ordinal` confirms the problem:** training has only the values 0, 1, 2, validation only 2, test only 3. The models use the other 19 features.
* **Missing values (training):** the championship positions `pos_before` (6.6%) and `cons_pos_before` (5.5%) are empty in round 1; the qualifying gap and lap time are empty for 1.3% (no timed lap); `qual_position` for 0.35%. P3's flags `standings_missing` (0.85%) and `cons_standings_missing` (0.24%) are themselves empty in a few rows, which a 0/1 flag should never be; the imputer fills them with the training median (0) and adds a missing-value flag, and we reported it to P3.

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation), **PR-AUC = the podium rate** (no better than random ranking) and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)")
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate (0.151 on validation) and F1 = 0, exactly as expected, so the function computes the metrics correctly. Its accuracy is still 84.9%, although it never finds a single podium. That is why accuracy is not one of our metrics, and why **PR-AUC is the natural primary metric** for this rare positive class. These numbers are the floor every real model must beat.

### 4.4 Baselines

A baseline is the simplest reasonable approach. Every later model must beat it, otherwise its extra complexity is not earned.
* **Rule `grid <= 3`:** predict a podium for the three cars that start at the front. It learns nothing; it encodes the domain fact that the starting position is the strongest single predictor.
* **Logistic regression:** the simplest statistical model. It multiplies each feature by a learned weight, adds them up, and turns the sum into a probability with the sigmoid function. *Limitation:* it is linear, so it cannot learn interactions on its own (for example "a good grid slot matters more for a top team").

#### Attempt 1: rule `grid <= 3`
The rule gives a hard yes/no, so its ROC-AUC and PR-AUC are those of a single operating point. It uses P3's `grid_fixed`, where a pit-lane start is slot 25, so a pit-lane start is never a front-row start.

In [ ]:
class GridRule:
    """Rule baseline: 'podium' if the car starts in grid slot 1, 2 or 3. Learns nothing."""
    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        p = X["grid_fixed"].between(1, 3).astype(float).to_numpy()
        return np.column_stack([1 - p, p])

rule_results = train_eval(GridRule(), X_train, y_train, X_val, y_val, name="rule grid <= 3")
display(rule_results)

#### Attempt 2: logistic regression (19 features)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_results = train_eval(lr, X_train, y_train, X_val, y_val, name="logistic regression")
display(lr_results)

#### Attempt 3: the same logistic regression with `era_ordinal` (all 20 of P3's features)
The evidence for leaving `era_ordinal` out (4.2). Same model, same rows, one column more. We also look at the weight the model learns for it and at how far the test value (3) lies outside the training values. This check uses only the training years, the validation years, and the *value* of the column in the test years, never a test label.

In [ ]:
prep_all = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())   # no column selection
lr_era = make_pipeline(prep_all, LogisticRegression(max_iter=2000, random_state=SEED))
lr_era_results = train_eval(lr_era, X_train, y_train, X_val, y_val, name="logistic regression + era_ordinal")
display(lr_era_results)

era_weight = pd.Series(lr_era[-1].coef_[0], index=lr_era[0].get_feature_names_out())["era_ordinal"]
scaler = lr_era[0][-1]
era_pos = list(lr_era[0][0].feature_names_in_).index("era_ordinal")
era_mean, era_sd = scaler.mean_[era_pos], scaler.scale_[era_pos]
z_test = (3 - era_mean) / era_sd
z_train_max = (X_train["era_ordinal"].max() - era_mean) / era_sd
print(f"validation PR-AUC: without era_ordinal {lr_results.iloc[1]['PR-AUC']:.4f}, "
      f"with era_ordinal {lr_era_results.iloc[1]['PR-AUC']:.4f}")
print(f"learned weight of era_ordinal (standardized input): {era_weight:+.3f}")
print(f"largest standardized era value seen in training: {z_train_max:+.2f}; the test value 3 becomes {z_test:+.2f}")
print(f"-> on every test row the score moves by {era_weight * (z_test - z_train_max):+.3f} beyond anything seen in training")

**What we see.**
* **On validation `era_ordinal` adds nothing:** PR-AUC 0.7376 without it and 0.7371 with it (-0.0005).
* **But it would change every test prediction:** the model learns a weight of -0.134 for it. In standardized units the largest era value in training is +1.01, while the test value 3 becomes +2.33. So on every test row the model's score would move by -0.18 in a region it has never seen, and no validation row (all era 2) can tell us whether that move is right.
* **Decision:** `era_ordinal` stays out of the models (it stays in P3's table for reporting). Era sensitivity is measured instead, per test season, in 4.12.

#### Attempt 4: logistic regression with `class_weight="balanced"`
Podiums are only about 14% of the training rows. `class_weight="balanced"` gives the podium rows a larger weight, so both classes count equally in training. We expect the ranking metrics (ROC-AUC, PR-AUC) to stay about the same and the yes/no metrics at threshold 0.5 to change, because the weighting mostly moves the predicted probabilities up.

In [ ]:
lr_bal = with_preprocessing(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
lr_bal_results = train_eval(lr_bal, X_train, y_train, X_val, y_val, name="logistic regression (balanced)")
display(lr_bal_results)

#### Attempt 5: logistic regression on the 4 draft features only
Before P3's table was ready, the models were built on four raw pre-race columns (`grid`, `pitlane_start`, `has_qualifying`, `qual_position`). Same model, same rows: does P3's feature engineering add anything?

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]
lr_draft = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, random_state=SEED))
lr_draft_results = train_eval(lr_draft, train[DRAFT_FEATURES].astype(float), y_train,
                              val[DRAFT_FEATURES].astype(float), y_val, name="logistic regression (4 draft features)")
display(lr_draft_results)

#### Comparison of the baselines (validation)

In [ ]:
import matplotlib.pyplot as plt

baselines = pd.concat([dummy_results, rule_results, lr_results, lr_era_results, lr_bal_results, lr_draft_results],
                      ignore_index=True)
val_table = baselines[baselines["split"] == "validation"].set_index("model").drop(columns="split")
display(val_table)

ax = val_table[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(figsize=(8, 4.5))
ax.set_title("Baselines on validation (2020-2021), threshold 0.5")
ax.set_xlabel("score (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="metric", loc="lower right")
plt.tight_layout()
plt.show()

# what the logistic regression learned: weights of the standardized inputs, largest first
weights = pd.Series(lr.named_steps["model"].coef_[0], index=lr.named_steps["prep"].get_feature_names_out())
display(weights.sort_values(key=abs, ascending=False).head(10).round(3).to_frame("weight (standardized input)"))

**What we see (validation 2020-2021).**
* **Logistic regression is far above both references:** PR-AUC 0.738 vs 0.530 for the rule and 0.151 for the dummy; ROC-AUC 0.931 vs 0.820.
* **P3's features add real information:** the same logistic regression on the same rows reaches PR-AUC 0.673 with the 4 draft features and **0.738 with the 19 features (+0.065)**. Which group brings how much is measured by P3's ablation and by the XAI in 4.11.
* **At threshold 0.5 the rule still has the best F1** (0.695 vs 0.685): plain logistic regression is cautious at 0.5 (precision 0.74, recall 0.64). The threshold matters as much as the model, so it is tuned in 4.8.
* **`class_weight="balanced"` behaves as expected:** PR-AUC barely changes (0.733 vs 0.738) because the ranking stays the same, while at 0.5 recall rises to 0.90 and precision falls to 0.45. Weighting mostly moves the probabilities, which threshold tuning can also do.
* **Weights:** the largest are `grid_fixed` (-1.03), `qual_position` (-0.95), `gap_to_pole_pct` (-0.81) and `pos_before` (-0.71): starting nearer the front, qualifying closer to pole and being higher in the championship all mean a higher podium probability. Twin features partly cancel each other, for example `team_circuit_podiums_before` (+0.30) and `team_circuit_starts_before` (-0.29), so the sign of a single weight should not be over-read. These are associations the model uses, not causes.
* **Train PR-AUC is a little lower than validation** (0.698 vs 0.738): the training years include less predictable seasons, while 2020-2021 were dominated by two teams.

**Baseline to beat:** logistic regression, validation PR-AUC **0.738**.

### 4.5 Statistical models: random forest and gradient boosting

Two tree-based models, chosen because they can learn what logistic regression cannot: non-linear effects and interactions between features (for example "a good grid slot matters more for a team that leads the championship"). Both are in scikit-learn, so no extra library is needed.
* **Random forest:** many decision trees, each trained on a random sample of rows and features; the prediction is the average vote. Averaging many different trees reduces overfitting. *Limitation:* its probabilities are averages of votes (not calibrated), and it cannot extrapolate beyond the values seen in training.
* **Gradient boosting** (`HistGradientBoostingClassifier`): small trees built one after another, each one correcting the errors of the trees before it. Usually the strongest model family on tables like ours. *Limitation:* more hyperparameters, and it overfits if the trees are too big or too many. With fewer than 10,000 training rows scikit-learn does **not** stop early by default, so it always builds `max_iter = 100` trees (the number of trees is studied in Attempt 19).

**Class imbalance:** both use `class_weight="balanced"` (podiums are about 14% of rows). Section 4.4 showed this mostly moves the probabilities; the threshold is tuned in 4.8. Hyperparameters are tuned in 4.7.

#### Attempt 6: random forest
300 trees; `min_samples_leaf=20` so every leaf is based on at least 20 driver-races, which keeps the probabilities smooth instead of memorizing single rows.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = with_preprocessing(RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                               n_jobs=-1, random_state=SEED))
rf_results = train_eval(rf, X_train, y_train, X_val, y_val, name="random forest")
display(rf_results)

#### Attempt 7: gradient boosting
Default settings for now.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = with_preprocessing(HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED))
gb_results = train_eval(gb, X_train, y_train, X_val, y_val, name="gradient boosting")
display(gb_results)

#### Two other model families (exploration)
The brief asks for two statistical models, and we have them. To check that we did not miss an easy win, we try two families that work in a completely different way, each with one sensible setting and no tuning. They are not candidates for the final model unless they clearly beat the others.
* **k-nearest neighbours (k-NN):** no training at all. For a new driver-race it finds the 50 most similar driver-races in the training years (similar grid, qualifying, standings...) and the probability is the share of them that ended on the podium. *Limitation:* every feature counts equally in "similar", even the weak ones, and it cannot extrapolate.
* **Gaussian Naive Bayes:** for each class it learns a bell curve per feature and multiplies them, assuming the features are **independent** of each other. *Limitation:* our features are strongly linked (grid and qualifying position, points and championship position), so the same evidence is counted several times.

#### Attempt 8: k-nearest neighbours (k = 50)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = with_preprocessing(KNeighborsClassifier(n_neighbors=50))
knn_results = train_eval(knn, X_train, y_train, X_val, y_val, name="k-nearest neighbours (k=50)")
display(knn_results)

#### Attempt 9: Gaussian Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = with_preprocessing(GaussianNB())
nb_results = train_eval(nb, X_train, y_train, X_val, y_val, name="naive Bayes")
display(nb_results)

#### Comparison of all models so far
Validation scores for every attempt, plus the train-validation gap in PR-AUC (a large positive gap means overfitting).

In [ ]:
def compare(results):
    """Validation scores of every model, plus the train - validation PR-AUC gap."""
    allres = pd.concat(results, ignore_index=True)
    pr = allres.pivot(index="model", columns="split", values="PR-AUC")
    summary = allres[allres["split"] == "validation"].set_index("model").drop(columns="split")
    summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
    return summary.sort_values("PR-AUC", ascending=False)

summary = compare([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results,
                   rf_results, gb_results, knn_results, nb_results])
display(summary)

lr_pr = summary.loc["logistic regression", "PR-AUC"]
ax = summary["PR-AUC"].sort_values().plot.barh(figsize=(8, 4.5), color="tab:orange")
ax.axvline(lr_pr, color="black", linestyle="--", label=f"baseline to beat (logistic regression) = {lr_pr:.4f}")
ax.set_title("PR-AUC on validation (2020-2021), all models so far")
ax.set_xlabel("PR-AUC (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What we see (validation 2020-2021).**
* **The tree models only tie logistic regression:** PR-AUC random forest 0.739, logistic regression 0.738, gradient boosting 0.728; ROC-AUC 0.929-0.934 for all three.
* **Gradient boosting with default settings overfits:** train PR-AUC 0.975 vs validation 0.728 (gap +0.25). With about 800 training podiums, trees with 31 leaves learn details of the training seasons that do not repeat. The random forest (`min_samples_leaf=20`) overfits much less (gap +0.05). Tuning (4.7) controls the tree size.
* **k-NN (0.677) is clearly worse** and no better than logistic regression on the 4 draft features: it treats all 19 features as equally important when it looks for "similar" driver-races, so the weak features blur the strong ones.
* **Naive Bayes is the worst real model (0.292, only twice the dummy).** It multiplies the evidence of features that say the same thing (grid, qualifying position, gap to pole...), so its probabilities are pushed to the extremes: at 0.5 it says "podium" for almost everyone (recall 0.99, precision 0.16). This is the independence assumption failing on our correlated features.
* **Why no gain from trees:** the strongest signal is a smooth "the nearer the front, the likelier the podium" relation that a straight line already captures; the extra interactions the trees can learn are small compared with the noise of 117 validation podiums.

**Decision:** random forest and gradient boosting stay as our two statistical models and are tuned in 4.7. k-NN and Naive Bayes are not tuned further: both are far behind, and their weaknesses (equal feature weights, independence) come from how they work, not from a setting. The baseline to beat is still logistic regression.

### 4.6 Shallow neural network (PyTorch)

**How it works:** a feed-forward network. Each hidden layer computes weighted sums of its inputs and passes them through a ReLU (negative values become 0), which lets the network learn non-linear patterns. The output layer gives one score that the sigmoid turns into a podium probability. The weights are learned by gradient descent (Adam) on the binary cross-entropy loss. With no hidden layer this would be exactly logistic regression, so the hidden layers are what the network adds.

**Choices** ("shallow" = 1-2 hidden layers, as the brief asks):
* **Dropout 0.2:** during training, 20% of the hidden units are switched off at random in every batch, so the network cannot rely on single units (less overfitting).
* **Early stopping on validation:** after every epoch (one pass over the training rows) we compute the validation loss; training stops when it has not improved for 15 epochs, and the weights of the best epoch are kept.
* **Class imbalance:** `pos_weight` = (non-podium rows / podium rows) in the loss, the same idea as `class_weight="balanced"`.
* **Preprocessing:** the same pipeline as the other models (4.2), fitted on the training rows only inside `fit()`.
* **Reproducible:** `torch.manual_seed(SEED)`, a seeded batch order and a single CPU thread (the number of threads changes the rounding of the sums), so Run All gives the same numbers on a laptop and on Kaggle.

*Limitations:* many settings (layers, units, dropout, learning rate), needs scaling and imputation, a black box without XAI, and on small tables it rarely beats simpler models. **Caveat:** the validation set chooses the stopping epoch, so the network's validation score is slightly optimistic compared with the other models; the test set gives the fair comparison.

The class below only wraps the network with `fit` / `predict_proba`, so the same `train_eval()` scores it.

In [ ]:
import torch
from torch import nn


class ShallowNN:
    """Feed-forward network wrapped with fit / predict_proba so train_eval() can score it.
    - hidden: the units of each hidden layer, e.g. (32,) = one layer, (64, 32) = two layers
    - preprocessing: the shared pipeline of 4.2, fitted on the training rows only, inside fit()
    - early stopping: training stops when the validation loss has not improved for `patience` epochs,
      and the weights of the best epoch are kept"""

    def __init__(self, X_val, y_val, hidden=(32,), dropout=0.2, lr=1e-3, batch_size=256,
                 max_epochs=200, patience=15, seed=SEED):
        self.X_val, self.y_val = X_val, y_val
        self.hidden, self.dropout, self.lr, self.batch_size = hidden, dropout, lr, batch_size
        self.max_epochs, self.patience, self.seed = max_epochs, patience, seed

    def _build(self, n_inputs):
        layers, size = [], n_inputs
        for h in self.hidden:                                   # hidden layer: linear -> ReLU -> dropout
            layers += [nn.Linear(size, h), nn.ReLU(), nn.Dropout(self.dropout)]
            size = h
        layers.append(nn.Linear(size, 1))                       # output: one score (logit) for "podium"
        return nn.Sequential(*layers)

    def _tensor(self, X):
        return torch.tensor(self.prep.transform(X), dtype=torch.float32)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        torch.set_num_threads(1)                                # same numbers on every machine
        self.prep = clone(preprocess).fit(X)
        Xt, yt = self._tensor(X), torch.tensor(np.asarray(y), dtype=torch.float32)
        Xv, yv = self._tensor(self.X_val), torch.tensor(np.asarray(self.y_val), dtype=torch.float32)

        self.net = self._build(Xt.shape[1])
        self.n_weights = sum(p.numel() for p in self.net.parameters())
        pos_weight = (yt == 0).sum() / (yt == 1).sum()          # class imbalance, like class_weight="balanced"
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(self.net.parameters(), lr=self.lr)
        gen = torch.Generator().manual_seed(self.seed)

        self.history = {"train": [], "validation": []}
        best_loss, best_state, waited = float("inf"), None, 0
        for epoch in range(self.max_epochs):
            self.net.train()                                   # dropout on
            batch_losses = []
            for idx in torch.randperm(len(Xt), generator=gen).split(self.batch_size):
                optimizer.zero_grad()
                loss = loss_fn(self.net(Xt[idx]).squeeze(1), yt[idx])
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            self.net.eval()                                    # dropout off
            with torch.no_grad():
                val_loss = loss_fn(self.net(Xv).squeeze(1), yv).item()
            self.history["train"].append(float(np.mean(batch_losses)))
            self.history["validation"].append(val_loss)

            if val_loss < best_loss:                           # early stopping on the validation loss
                best_loss, waited = val_loss, 0
                best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                self.best_epoch = epoch + 1
            else:
                waited += 1
                if waited >= self.patience:
                    break
        self.net.load_state_dict(best_state)                   # keep the best epoch, not the last one
        return self

    def predict_proba(self, X):
        self.net.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.net(self._tensor(X)).squeeze(1)).numpy()
        return np.column_stack([1 - p, p])


def run_network(hidden, name):
    """Train one network on the training years, score it, and print its size and stopping epoch."""
    net = ShallowNN(X_val, y_val, hidden=hidden)
    res = train_eval(net, X_train, y_train, X_val, y_val, name=name)
    display(res)
    print(f"{net.n_weights} weights; stopped after {len(net.history['train'])} epochs, best epoch: {net.best_epoch}")
    return net, res

#### Attempt 10: one hidden layer (32 units)

In [ ]:
nn1, nn1_results = run_network((32,), "neural network (32)")

#### Attempt 11: two hidden layers (64 and 32 units)

In [ ]:
nn2, nn2_results = run_network((64, 32), "neural network (64-32)")

#### Attempt 12: a wider layer (one hidden layer, 128 units)
Same depth as Attempt 10 but four times the units. If the 32-unit layer were too small to hold the patterns, more units would help.

In [ ]:
nn128, nn128_results = run_network((128,), "neural network (128)")

#### Attempt 13: a deeper network (three hidden layers, 128-64-32)
One step beyond "shallow": if 1 and 2 layers are close, does a third layer find anything more? (Kept out of the tuning grid in 4.7, because the brief asks for a shallow network; this attempt only shows what depth does.)

In [ ]:
nn3, nn3_results = run_network((128, 64, 32), "neural network (128-64-32)")

#### Loss curves
Training loss (average over the batches of each epoch, with dropout on) and validation loss per epoch. The dashed line marks the epoch whose weights were kept.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, net, title in [(axes[0, 0], nn1, "1 hidden layer (32)"), (axes[0, 1], nn2, "2 hidden layers (64-32)"),
                       (axes[1, 0], nn128, "1 wide hidden layer (128)"), (axes[1, 1], nn3, "3 hidden layers (128-64-32)")]:
    epochs = range(1, len(net.history["train"]) + 1)
    ax.plot(epochs, net.history["train"], label="training loss")
    ax.plot(epochs, net.history["validation"], label="validation loss")
    ax.axvline(net.best_epoch, color="black", linestyle="--", label=f"best epoch ({net.best_epoch})")
    ax.set_title(f"Neural network, {title}")
    ax.set_xlabel("epoch")
    ax.legend()
for ax in axes[:, 0]:
    ax.set_ylabel("weighted binary cross-entropy")
plt.tight_layout()
plt.show()

#### Comparison of the networks (validation)

In [ ]:
nets = compare([nn1_results, nn2_results, nn128_results, nn3_results])
nets.insert(0, "weights", [{"neural network (32)": nn1, "neural network (64-32)": nn2, "neural network (128)": nn128,
                             "neural network (128-64-32)": nn3}[m].n_weights for m in nets.index])
display(nets)

**What we see (validation 2020-2021).**
* **All four networks land close to logistic regression (0.738):** 0.745 with one layer of 32, **0.754 with two layers (64-32)**, 0.737 with one wide layer of 128, 0.744 with three layers.
* **Wider is not better (Attempt 12):** four times the units (3,585 weights instead of 897) gives a *lower* validation PR-AUC (0.737 vs 0.745). The pattern in our data is simple, so the extra units mostly have more room to learn details of the training seasons.
* **Deeper is not better (Attempt 13):** the third layer gives 0.744, no better than two layers, with 3.6 times their weights (13,825). Its loss curve shows why: it reaches its best validation loss already at epoch 18, then the validation loss turns up and gets noisy while the training loss keeps falling, the start of overfitting.
* **All four are within 0.017 of each other,** far less than the noise of the validation PR-AUC (standard error about 0.038, measured in 4.9). So the honest answer to "why not a bigger or deeper network?" is: we tried, and with this table size does not matter; a shallow network is enough.
* **Loss curves:** both losses fall quickly in the first 5-10 epochs and then flatten; early stopping keeps the best epoch (74, 34, 49 and 18). The training loss looks lower partly because it is measured with dropout on and as an average over the epoch.
* **At threshold 0.5** the networks behave like the other balanced models: high recall (0.86-0.91), precision about 0.50.

### 4.7 Hyperparameter tuning (validation only)

For each model we try a small grid of the settings that control its complexity. Every combination is trained on the training years and scored on validation with `train_eval()`; the best combination is the one with the highest **validation PR-AUC** (our primary metric, see 4.8). The test years are not touched.

The grids are deliberately small: the validation set has only 117 podiums, so a large search would mostly fit its noise.

In [ ]:
from sklearn.model_selection import ParameterGrid

def tune(name, make_model, grid):
    """Train one model per combination in `grid` (training years) and score it on validation.
    Returns the fitted model with the best validation PR-AUC and the table of all results."""
    rows, best_model, best_pr = [], None, -1
    for params in ParameterGrid(grid):
        candidate = make_model(**params)
        res = train_eval(candidate, X_train, y_train, X_val, y_val, name=name).set_index("split")
        rows.append({**params, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"], "val ROC-AUC": res.loc["validation", "ROC-AUC"]})
        if res.loc["validation", "PR-AUC"] > best_pr:
            best_model, best_pr = candidate, res.loc["validation", "PR-AUC"]
    return best_model, pd.DataFrame(rows).sort_values("val PR-AUC", ascending=False, ignore_index=True)

tuned = {}   # model name -> (best fitted model, table of all combinations)

#### Attempt 14: tuning logistic regression
`C` is the inverse regularization strength: a small `C` pushes the weights towards 0 (simpler model), a large `C` lets them grow.

In [ ]:
tuned["logistic regression"] = tune(
    "logistic regression",
    lambda C: with_preprocessing(LogisticRegression(C=C, max_iter=2000, random_state=SEED)),
    {"C": [0.01, 0.1, 1, 10]})
display(tuned["logistic regression"][1])

#### Attempt 15: tuning the random forest
`min_samples_leaf`: the minimum number of driver-races behind each leaf (larger = smoother, less memorizing). `max_features`: how many inputs each split may look at (`"sqrt"` = a random subset, `1.0` = all of them; a subset makes the trees more different from each other).

In [ ]:
tuned["random forest"] = tune(
    "random forest",
    lambda min_samples_leaf, max_features: with_preprocessing(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=min_samples_leaf, max_features=max_features,
                               class_weight="balanced", n_jobs=-1, random_state=SEED)),
    {"min_samples_leaf": [5, 20, 50], "max_features": ["sqrt", 1.0]})
display(tuned["random forest"][1])

#### Attempt 16: tuning gradient boosting
`learning_rate`: how much each new tree may correct the previous ones (smaller = slower, more careful). `max_leaf_nodes`: the size of each tree (smaller = simpler trees).

In [ ]:
tuned["gradient boosting"] = tune(
    "gradient boosting",
    lambda learning_rate, max_leaf_nodes: with_preprocessing(HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, class_weight="balanced", random_state=SEED)),
    {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [7, 15, 31]})
display(tuned["gradient boosting"][1])

#### Attempt 17: tuning the neural network
`hidden`: the size of the hidden layer(s), including the 128-unit layer from Attempt 12. `dropout`: the share of hidden units switched off during training. (The number of epochs is already chosen by early stopping.)

In [ ]:
tuned["neural network"] = tune(
    "neural network",
    lambda hidden, dropout: ShallowNN(X_val, y_val, hidden=hidden, dropout=dropout),
    {"hidden": [(16,), (32,), (128,), (64, 32)], "dropout": [0.2, 0.4]})
display(tuned["neural network"][1])

**What we see (tuning, validation PR-AUC).**
* **Logistic regression:** best with `C=0.1` (0.741, vs 0.738 with the default `C=1`); even strong regularization (`C=0.01`) only costs 0.005. With 19 features and 5,742 rows the model is not starved of data.
* **Random forest:** best with `max_features="sqrt"` and `min_samples_leaf=20` (0.739), the setting of Attempt 6. Small leaves memorize the training seasons (`min_samples_leaf=5`: train 0.927, validation 0.731), and letting every split see all features (`1.0`) makes the trees too similar (0.704-0.725).
* **Gradient boosting:** best with the **smallest trees and the slowest learning** (`max_leaf_nodes=7`, `learning_rate=0.03`): 0.736 (0.728 with the defaults), and the overfitting of 4.5 disappears (train 0.718). Big trees overfit (31 leaves: train 0.975).
* **Neural network:** best with two hidden layers (64-32) and dropout 0.2: 0.754, the setting of Attempt 11. The 128-unit layer does better with more dropout (0.743 with 0.4 vs 0.737 with 0.2), a sign that it has more capacity than the data needs, but it is still not the best.
* **Tuning gains are small** (at most +0.009, for gradient boosting), and the four tuned models end within 0.018 of each other (0.736-0.754), less than the noise of the validation PR-AUC.

#### Attempt 18: how many trees? (random forest and gradient boosting)
Until now the forest always had 300 trees and boosting 100. Here we keep each model's tuned settings and change only the number of trees, scoring train and validation every time. The two families should react differently: a forest **averages** independent trees, so more trees can only make the average steadier; boosting **adds** trees that keep correcting the training errors, so too many trees start to learn the noise of the training years.

In [ ]:
best_rf = tuned["random forest"][1].iloc[0]          # tuned settings from Attempts 15 and 16
best_gb = tuned["gradient boosting"][1].iloc[0]
tree_counts = [10, 25, 50, 100, 200, 400, 800]

rows = []
for n in tree_counts:
    forest = with_preprocessing(RandomForestClassifier(
        n_estimators=n, min_samples_leaf=int(best_rf["min_samples_leaf"]), max_features=best_rf["max_features"],
        class_weight="balanced", n_jobs=-1, random_state=SEED))
    boosting = with_preprocessing(HistGradientBoostingClassifier(
        max_iter=n, early_stopping=False, learning_rate=best_gb["learning_rate"],
        max_leaf_nodes=int(best_gb["max_leaf_nodes"]), class_weight="balanced", random_state=SEED))
    for family, clf in [("random forest", forest), ("gradient boosting", boosting)]:
        res = train_eval(clf, X_train, y_train, X_val, y_val, name=family).set_index("split")
        rows.append({"model": family, "trees": n, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"]})
trees = pd.DataFrame(rows)
display(trees.pivot(index="trees", columns="model", values=["train PR-AUC", "val PR-AUC"]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, family, used in [(axes[0], "random forest", 300), (axes[1], "gradient boosting", 100)]:
    part = trees[trees["model"] == family]
    ax.plot(part["trees"], part["train PR-AUC"], marker="o", label="train")
    ax.plot(part["trees"], part["val PR-AUC"], marker="o", label="validation")
    ax.axvline(used, color="black", linestyle="--", label=f"number used in the models ({used})")
    ax.set_xscale("log")
    ax.set_xticks(tree_counts, [str(n) for n in tree_counts])
    ax.set_title(f"{family}: PR-AUC vs number of trees")
    ax.set_xlabel("number of trees (log scale)")
    ax.legend()
axes[0].set_ylabel("PR-AUC")
plt.tight_layout()
plt.show()

**What we see (number of trees).**
* **Random forest: more trees never hurt, they only steady the average.** Validation PR-AUC goes from 0.724 with 10 trees to about 0.74 with 200 and then stays flat (0.738-0.741 up to 800 trees); the training score is flat too (about 0.79). The 300 trees we use are on the plateau; more would only cost time.
* **Gradient boosting: the number of trees is a complexity knob.** The training score keeps rising (0.640 with 10 trees to 0.857 with 800) while validation peaks at 200 trees (0.746) and then falls (0.729 at 400, 0.721 at 800): every extra tree corrects the remaining training errors, and after a point those errors are noise.
* **We keep the tuned models unchanged.** 200 boosting trees would add +0.010 on validation, about a quarter of the noise level, and it would not change the final model (4.9). Picking the tree count from this curve would be one more choice fitted to 117 validation podiums. The useful lesson is the shape: for boosting the number of trees must be limited (or stopped early), for a forest it does not matter.

### 4.8 Metric choice and decision threshold

**Primary metric: PR-AUC.** Podiums are rare (about 15% of the validation rows). PR-AUC measures how well the real podium finishers are ranked at the top, it does not depend on a threshold, and its floor is the podium rate (the dummy in 4.3) instead of 0.5, so it shows clearly how much a model adds.

**Secondary metric: F1 at the chosen threshold.** In the end the model must say yes or no; F1 balances precision (how many predicted podiums are right) and recall (how many real podiums are found).

ROC-AUC is reported for completeness, but with about 85% non-podium rows it looks good easily (about 0.8 even for the simple rule).

**Threshold:** for every tuned model we try thresholds from 0.05 to 0.95 (step 0.01) on validation and keep the one with the highest F1. Each model gets its own threshold, because class weighting shifts the probabilities (4.4).

In [ ]:
def best_threshold(clf, X, y):
    """Threshold between 0.05 and 0.95 (step 0.01) with the highest F1 on (X, y)."""
    proba = clf.predict_proba(X)[:, 1]
    candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
    f1s = [f1_score(y, proba >= t, zero_division=0) for t in candidates]
    return candidates[int(np.argmax(f1s))]

rows = []
for name, (clf, _) in tuned.items():
    t = best_threshold(clf, X_val, y_val)
    rows.append({"model": name, "threshold": t, **evaluate(clf, X_val, y_val, threshold=t)})
tuned_table = pd.DataFrame(rows).set_index("model").round(4)
thresholds = tuned_table["threshold"].to_dict()
display(tuned_table)

### 4.9 Choosing the final model (validation only)

Before comparing the models we measure how noisy the validation PR-AUC is: we resample the validation rows with replacement 1000 times (bootstrap) and take the standard deviation of PR-AUC as its standard error (SE).

**Rule (one-standard-error rule):** among the models whose validation PR-AUC is within one SE of the best, take the simplest (order: logistic regression < random forest < gradient boosting < neural network). A difference smaller than the noise is not a reason to accept a more complex model. The final model and its threshold are then used once on the test set (4.10) and by P1's inference function (4.13).

In [ ]:
COMPLEXITY = ["logistic regression", "random forest", "gradient boosting", "neural network"]   # simplest first
val_pr = tuned_table["PR-AUC"]
best_name = val_pr.idxmax()

# bootstrap standard error of the best model's validation PR-AUC
rng = np.random.default_rng(SEED)
proba, y = tuned[best_name][0].predict_proba(X_val)[:, 1], y_val.to_numpy()
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    boot.append(average_precision_score(y[i], proba[i]))
se = np.std(boot)

within_one_se = [name for name in COMPLEXITY if val_pr[name] >= val_pr[best_name] - se]
FINAL_NAME = within_one_se[0]
FINAL_MODEL, FINAL_THRESHOLD = tuned[FINAL_NAME][0], thresholds[FINAL_NAME]

print(f"best validation PR-AUC: {best_name} = {val_pr[best_name]:.4f}, bootstrap SE = {se:.4f} "
      f"(95% range {np.percentile(boot, 2.5):.3f}-{np.percentile(boot, 97.5):.3f})")
print(f"models within one SE: {within_one_se}")
print(f"final model: {FINAL_NAME}, threshold {FINAL_THRESHOLD}")

**How the final threshold was chosen:** precision, recall and F1 of the final model on validation for every threshold.

In [ ]:
proba = FINAL_MODEL.predict_proba(X_val)[:, 1]
candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([evaluate(FINAL_MODEL, X_val, y_val, threshold=t) for t in candidates], index=candidates)

ax = curve[["precision", "recall", "F1"]].plot(figsize=(8, 4))
ax.axvline(FINAL_THRESHOLD, color="black", linestyle="--", label=f"chosen threshold = {FINAL_THRESHOLD}")
ax.set_title(f"Final model ({FINAL_NAME}): metrics vs threshold on validation")
ax.set_xlabel("threshold (predict podium if probability >= threshold)")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

# compare the final yes/no decisions with the simple rule grid <= 3 from 4.4
rule_pred = X_val["grid_fixed"].between(1, 3).to_numpy()
disagree = int(((proba >= FINAL_THRESHOLD) != rule_pred).sum())
print(f"rows where the final model and the rule grid <= 3 disagree: {disagree} of {len(X_val)}")
display(curve.loc[sorted({0.2, 0.3, 0.4, FINAL_THRESHOLD, 0.5, 0.6})].round(3))     # how flat F1 is around the choice

**What we see (threshold and final model).**
* **Each model needs its own threshold:** 0.42 for logistic regression, 0.65 for the random forest, 0.70 for gradient boosting, 0.78 for the network. The balanced models have their probabilities pushed up by the class weights, so their best threshold is higher.
* **With its own threshold every model reaches a validation F1 of about 0.71-0.74** (gradient boosting 0.739, network 0.724, random forest 0.718, logistic regression 0.711).
* **Final model:** the bootstrap standard error of the validation PR-AUC is 0.038 (95% range 0.68-0.83). All four models are within one standard error of the best (the network, 0.754), so the one-standard-error rule picks the simplest: **logistic regression (`C=0.1`) with threshold 0.42**.
* **The threshold sits on a plateau:** validation F1 is 0.708 at 0.30, 0.701 at 0.40, 0.711 at 0.42 and 0.682 at 0.50, then it falls quickly (0.608 at 0.60). Anything between about 0.3 and 0.45 is almost equally good; 0.42 is the point of that plateau with higher precision (0.73) than recall (0.69). Because the curve is this flat, the exact best threshold moves between table versions (it was 0.23 on P3's previous table), while the F1 hardly changes.
* **More than the rule:** the final model disagrees with the rule `grid <= 3` on 31 of 774 validation rows, so it does not just repeat "the front three on the grid".

### 4.10 Final evaluation on the test set (used once)

Until now the test years (2022-2024) were never used in this run: every model, hyperparameter, threshold and the final choice were fixed on the training and validation years. Here each tuned model is scored **once** on the test set with the threshold it got on validation, next to its train and validation scores, so we can see whether the validation results hold on unseen seasons. Nothing is changed after this point. The rule `grid <= 3` is included as the reference to beat.

**Honest note on the test set.** This part was run on three versions of P3's table. (1) 18 features with a one-hot `era`: the test probabilities were pushed up (about 5 predicted podiums per race instead of 3), because the test era never occurs in training; P3 then removed `era`. (2) 17 features, training from 2003. (3) This run: P3's current table, training from 2006, 20 features, of which we use 19. Every change of the feature table was made by P3 for reasons she documents on the training and validation years, and our exclusion of `era_ordinal` is justified without test labels (4.2 and Attempt 3). **Nothing in this part was chosen on the test set:** the models, the hyperparameter grids, the threshold rule and the final-model rule are the same in every run, and every choice is made on validation.

In [ ]:
test = table[table["split"] == "test"]
X_test, y_test = test[FEATURES].astype(float), test["podium"]
print(f"test: {len(X_test)} rows ({test['year'].min()}-{test['year'].max()}), {y_test.sum()} podiums, "
      f"podium rate {y_test.mean():.3f} (= the PR-AUC of a model that knows nothing)")

final_models = {"rule grid <= 3": (GridRule(), 0.5),
                **{name: (clf, thresholds[name]) for name, (clf, _) in tuned.items()}}
rows = []
for name, (clf, t) in final_models.items():
    label = f"{name} (final)" if name == FINAL_NAME else name
    for split, X, y in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
        rows.append({"model": label, "split": split, "threshold": t, **evaluate(clf, X, y, threshold=t)})
comparison = pd.DataFrame(rows).round(4)
display(comparison)

# the two metrics we chose (4.8), side by side per split
display(comparison.pivot(index="model", columns="split", values=["PR-AUC", "F1"])
        .reindex(columns=["train", "validation", "test"], level=1))

#### ROC and precision-recall curves (test set)
The ROC curve shows the true-positive rate against the false-positive rate for every threshold; the precision-recall curve shows precision against recall. The area under each curve is the ROC-AUC and the PR-AUC (`AP`) in the table.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))
for name, (clf, _) in tuned.items():
    proba = clf.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax_pr)
ax_roc.plot([0, 1], [0, 1], color="black", linestyle="--", label="random guess (AUC = 0.5)")
ax_pr.axhline(y_test.mean(), color="black", linestyle="--", label=f"podium rate = {y_test.mean():.3f}")
ax_roc.set_title("ROC curves on the test set (2022-2024)")
ax_pr.set_title("Precision-recall curves on the test set (2022-2024)")
ax_roc.legend(loc="lower right")
ax_pr.legend(loc="lower left")
plt.tight_layout()
plt.show()

#### Confusion matrices (test set, threshold chosen on validation)
Rows are the real outcome, columns the prediction. Top-right = false alarms (predicted a podium that did not happen), bottom-left = missed podiums.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (clf, _)) in zip(axes, tuned.items()):
    pred = (clf.predict_proba(X_test)[:, 1] >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["no podium", "podium"],
                                            ax=ax, colorbar=False)
    ax.set_title(f"{name}{' (final)' if name == FINAL_NAME else ''}\nthreshold {thresholds[name]}")
plt.tight_layout()
plt.show()

**What we see (test set 2022-2024, 1,351 rows, 204 podiums).**
* **The final model ranks best on unseen seasons.** PR-AUC: logistic regression 0.702, network 0.688, random forest 0.685, gradient boosting 0.674; rule 0.474; no-skill floor 0.151. ROC-AUC is 0.93-0.94 for all four.
* **F1 is a tie:** random forest 0.680, logistic regression 0.673, gradient boosting and network 0.670, rule 0.649. The random forest's lead (+0.007) is far below the noise, and switching to it now would be a choice made on the test set, so we do not.
* **The one-standard-error rule paid off again:** from validation to test the network drops most (0.754 to 0.688, its stopping epoch was chosen on validation), gradient boosting by 0.063 and the forest by 0.054, while the simplest model drops least (0.741 to 0.702).
* **Final model in numbers (threshold 0.42):** it finds 146 of the 204 podiums (recall 0.72) with 84 false alarms (precision 0.63) and 58 missed podiums.
* **Compared with the previous run on P3's 17-feature table (training from 2003):** test PR-AUC 0.711 then, 0.702 now; F1 0.683 then, 0.673 now. Both differences are well inside the noise. The new table trains on fewer seasons (from 2006) and no longer has the recent-form features (P3 removed them; in the old run their group had a permutation importance of 0.028).
* **No choice was made on the test set:** the final model and threshold are the ones chosen on validation in 4.9.

### 4.11 Explainability (XAI) of the final model

We explain the final model chosen in 4.9 (`FINAL_MODEL`). The code only calls `predict_proba`, so it works for any of our models. We use the validation years: explanations do not choose anything, so there is no reason to touch the test set again. Only the 19 features the model uses are explained.

* **Permutation importance (global):** shuffle one feature at a time and measure how much the validation PR-AUC drops. A big drop means the model relies on that feature. Because several features carry the same information (for example `grid_fixed` and `qual_position`), we also shuffle each of P3's **feature groups** together: shuffling only one of two twin features underestimates both, since the model can lean on the twin.
* **SHAP (global and local):** splits each prediction into one contribution per feature, starting from a baseline (the average prediction). The contributions add up exactly to the predicted probability.
* **LIME (local):** for one row, creates many slightly changed copies, asks the model about them, and fits a small linear model around that row. Its weights say which feature values push this one prediction up or down.

**Not causal.** All three methods describe how *the model* uses the features to make predictions. A feature being important is not proof that it causes podium finishes: starting at the front does not by itself put a driver on the podium; a fast car and driver cause both the good grid slot and the podium, and the model only uses the grid slot as a signal of that.

#### Global: permutation importance (validation, 30 shuffles)

In [ ]:
from sklearn.inspection import permutation_importance

pr_auc_scorer = lambda clf, X, y: average_precision_score(y, clf.predict_proba(X)[:, 1])   # works for any of our models
pi = permutation_importance(FINAL_MODEL, X_val, y_val, scoring=pr_auc_scorer, n_repeats=30, random_state=SEED)
perm = pd.DataFrame({"PR-AUC drop": pi.importances_mean, "std": pi.importances_std}, index=FEATURES).loc[MODEL_FEATURES]

# the same idea per feature group: shuffle all columns of a group together
rng = np.random.default_rng(SEED)
full_score = pr_auc_scorer(FINAL_MODEL, X_val, y_val)
group_rows = []
for group, cols in MODEL_GROUPS.items():
    drops = []
    for _ in range(30):
        shuffled = X_val.copy()
        shuffled[cols] = X_val[cols].iloc[rng.permutation(len(X_val))].set_axis(X_val.index)
        drops.append(full_score - pr_auc_scorer(FINAL_MODEL, shuffled, y_val))
    group_rows.append({"group": group, "PR-AUC drop": np.mean(drops), "std": np.std(drops)})
group_perm = pd.DataFrame(group_rows).set_index("group")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [3, 2]})
for ax, imp, title in [(ax1, perm, "per feature"), (ax2, group_perm, "per feature group (shuffled together)")]:
    imp = imp.sort_values("PR-AUC drop")
    ax.barh(imp.index, imp["PR-AUC drop"], xerr=imp["std"], color="tab:blue")
    ax.set_title(f"Permutation importance {title}")
    ax.set_xlabel("drop in validation PR-AUC (bar = mean, line = std)")
fig.suptitle(f"Final model: {FINAL_NAME}")
plt.tight_layout()
plt.show()
display(perm.sort_values("PR-AUC drop", ascending=False).round(4).head(8))
display(group_perm.sort_values("PR-AUC drop", ascending=False).round(4))

#### Global: SHAP summary (validation)
SHAP and LIME call the model with plain number arrays, so a small helper `predict_podium_numeric` first turns them back into a table with the feature names. (Its name differs from P1's inference function `predict_podium` on purpose, so the two do not overwrite each other in the merged notebook.) Rows with a missing value (mostly round 1, without a championship position) are left out of the explanations, so every explained value is real. Each dot is one validation row. Position = how much that feature moved the predicted podium probability for that row (right = up, left = down); colour = the feature's value (red = high, blue = low).

In [ ]:
import warnings
with warnings.catch_warnings():          # shap's import prints a harmless progress-bar warning in notebooks
    warnings.simplefilter("ignore")
    import shap

def predict_podium_numeric(A):
    """Podium probability for a plain number array in MODEL_FEATURES order (how SHAP and LIME call the model).
    The columns the model does not use (era_ordinal) are added as empty; the model's first step drops them."""
    X = pd.DataFrame(np.asarray(A, dtype=float), columns=MODEL_FEATURES).reindex(columns=FEATURES)
    return FINAL_MODEL.predict_proba(X)[:, 1]

complete = val[MODEL_FEATURES].notna().all(axis=1)
xai_rows = val[complete]                                   # validation rows without missing values
X_xai, y_xai = xai_rows[MODEL_FEATURES].astype(float), xai_rows["podium"].to_numpy()
print(f"explained rows: {len(X_xai)} of {len(val)} validation rows")

background = X_xai.sample(100, random_state=SEED)          # SHAP's baseline = average prediction over these rows
explainer = shap.Explainer(predict_podium_numeric, shap.maskers.Independent(background), algorithm="permutation",
                           feature_names=MODEL_FEATURES, seed=SEED)
shap_values = explainer(X_xai, max_evals=2 * len(MODEL_FEATURES) + 1, silent=True)
print(f"SHAP baseline (average predicted podium probability of the background rows): {shap_values.base_values[0]:.3f}")

shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title(f"SHAP summary of the final model ({FINAL_NAME}), validation")
plt.xlabel("SHAP value (change in predicted podium probability)")
plt.tight_layout()
plt.show()

#### Local: one podium row and one non-podium row
Chosen by a fixed rule, not by hand: the podium finisher with the **highest** predicted probability (a clear case), and the non-podium row with the **median** predicted probability among non-podiums (a typical midfield car). The finishing position and status are shown only to tell the story; they are post-race information and never features.

In [ ]:
race_names = tables["races"].set_index("raceId")["name"]
driver_names = tables["drivers"].set_index("driverId")["surname"]
finish = outcomes.set_index(["raceId", "driverId"])[["positionOrder", "status"]]

proba_xai = predict_podium_numeric(X_xai)
non_podium_rows = np.where(y_xai == 0)[0]
examples = {
    "podium": int(np.argmax(np.where(y_xai == 1, proba_xai, -1))),
    "non-podium": int(non_podium_rows[np.argsort(proba_xai[non_podium_rows])[len(non_podium_rows) // 2]]),
}

def describe(i):
    """Who/where for explained row i (finishing position and status from `outcomes`, for the story only)."""
    r = xai_rows.iloc[i]
    pos, status = finish.loc[(r["raceId"], r["driverId"])]
    return (f"{driver_names[r['driverId']]}, {race_names[r['raceId']]} {r['year']}: grid {r['grid']}, "
            f"predicted {proba_xai[i]:.1%}, finished P{pos} ({status})")

for label, i in examples.items():
    print(f"{label:>10}: {describe(i)}")

**Local SHAP (waterfall):** starts at the baseline at the bottom and adds each feature's contribution until it reaches this row's predicted probability at the top.

In [ ]:
for label, i in examples.items():
    shap.plots.waterfall(shap_values[i], max_display=10, show=False)
    plt.title(f"Local SHAP, {label} example\n{describe(i)}")
    plt.tight_layout()
    plt.show()

**LIME:** the bars are the weights of LIME's small linear model around this row (green pushes towards podium, red away from it). We also print how close LIME's local model gets to the real model's probability.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

yes_no_columns = [j for j, c in enumerate(MODEL_FEATURES) if X_xai[c].nunique() <= 2]   # 0/1 features, treated as categories
lime_explainer = LimeTabularExplainer(X_xai.to_numpy(), feature_names=MODEL_FEATURES,
                                      class_names=["no podium", "podium"], categorical_features=yes_no_columns,
                                      mode="classification", random_state=SEED)
predict_both = lambda A: np.column_stack([1 - predict_podium_numeric(A), predict_podium_numeric(A)])

for label, i in examples.items():
    lime_result = lime_explainer.explain_instance(X_xai.iloc[i].to_numpy(), predict_both, num_features=10)
    lime_result.as_pyplot_figure()
    plt.title(f"LIME, {label} example\n{describe(i)}")
    plt.xlabel("LIME weight (effect on the podium probability)")
    plt.tight_layout()
    plt.show()
    print(f"{label}: model probability {proba_xai[i]:.3f}, LIME's local estimate {lime_result.local_pred[0]:.3f}, "
          f"local fit R^2 {lime_result.score:.2f}")

#### The most confident miss
The non-podium row the model was most sure about: what happened in that race?

In [ ]:
most_confident_miss = int(np.argmax(np.where(y_xai == 0, proba_xai, -1)))
print(f"most confident miss: {describe(most_confident_miss)}")

**Interpretation.**
* **Three signals carry the model: grid, qualifying and the championship.** Per feature, shuffling `grid_fixed` drops the validation PR-AUC by 0.123, `qual_position` by 0.108 and `pos_before` by 0.061; every other feature is below 0.03. Per group, **qualifying (0.189), driver standings (0.152) and grid (0.129)** come first; context (0.010), circuit history (0.006) and constructor standings (0.002) add almost nothing. P3's ablation (retraining without each group) gives the same top three and the same weak three.
* **Per feature vs per group:** the qualifying group drops more (0.189) than `qual_position` alone (0.108), because shuffling the whole group also removes the twins (`gap_to_pole_pct`, `best_qual_s`) the model could otherwise lean on. **Permutation vs ablation:** permutation keeps the trained model and breaks a group, while P3's ablation retrains without it, so the retrained model can lean on the remaining twins; that is why ablation drops are much smaller.
* **Direction (SHAP summary):** a low (blue) `grid_fixed`, `qual_position` and `pos_before` push the probability up by up to about +0.2; many points before the race (`pts_before`, red) push it up too. Many earlier team starts at the circuit push *down* while many earlier team podiums there push up: the model uses the pair as a team success rate at that track, so a single one of the two should not be read alone.
* **Podium example (Hamilton, Bahrain 2020, from pole, won):** SHAP starts at the baseline 0.175 and adds qualifying P1 (+0.19), grid 1 (+0.16), leading the championship (+0.15), 307 points (+0.14), gap to pole 0 (+0.06) and 8 earlier podiums at this circuit (+0.05), reaching 0.899. **Non-podium example (Tsunoda, United States 2021, grid 10, finished 9th):** 14th in the championship (-0.04), qualifying 10th (-0.04), grid 10 (-0.04), only 18 points (-0.02) and 2.2% off pole (-0.02) bring him down to 0.014. Both stories are what a fan would expect.
* **LIME agrees on the direction but is only an approximation:** its small linear model estimates 0.29 where the real model says 0.90 (local fit R² 0.43), and it fits the non-podium row poorly (R² 0.06). SHAP values add up exactly to the model's prediction, so we trust SHAP more for the size of each effect.
* **The most confident miss is Hamilton at the 2020 Italian Grand Prix:** pole position, predicted 86.1%, finished 7th after a stop-and-go penalty during the race. Events inside the race cannot be seen by a pre-race model; this is a limit of the task, not of the model (4.12).

**Not causal.** These explanations describe the model, not Formula 1. The model relies on the qualifying, grid and championship positions because they are the best pre-race summary of how fast the car and driver are; it does not show that the starting position *causes* podiums, and moving a driver up the grid would not give him the model's probability.

### 4.12 Model limitations

Four checks give numbers to the main limitations.
1. **Are the predicted probabilities realistic in every split?** Every race has exactly 3 podiums, so the predicted probabilities of one race should add up to about 3. This check needs **no labels**, so it can be run on any future season before its races; we also compare the average predicted probability with the real podium rate.
2. **Era sensitivity:** the test years are a new technical era. Is the final model equally good in each test season?
3. **The false alarms on the test set:** how many of those cars did not finish the race (crash, breakdown), something no pre-race model can see?
4. **Feature drift:** do the feature values of the test years look like those of the training years? This also needs no labels.

These checks only describe the final model; nothing is changed because of them.

In [ ]:
# 1) are the probabilities realistic? (podiums per race should be about 3; this part needs no labels)
calib_rows = []
for split, part in [("train", train), ("validation", val), ("test", test)]:
    p = pd.Series(FINAL_MODEL.predict_proba(part[FEATURES].astype(float))[:, 1], index=part.index)
    calib_rows.append({"split": split, "predicted podiums per race (median)": p.groupby(part["raceId"]).sum().median(),
                       "average predicted probability": p.mean(), "real podium rate": part["podium"].mean()})
display(pd.DataFrame(calib_rows).set_index("split").round(3))

# 2) era sensitivity: the final model on each test season separately
season_rows = []
for year, part in test.groupby("year"):
    season_rows.append({"season": year, "races": part["raceId"].nunique(),
                        **evaluate(FINAL_MODEL, part[FEATURES].astype(float), part["podium"], threshold=FINAL_THRESHOLD)})
display(pd.DataFrame(season_rows).set_index("season").round(3))

# 3) false alarms on the test set: how many of those cars did not finish the race?
status = test[["raceId", "driverId"]].merge(outcomes[["raceId", "driverId", "status"]],
                                            on=["raceId", "driverId"], how="left")["status"]
finished = (status.eq("Finished") | status.str.match(r"^\+\d+ Laps?$")).to_numpy()
said_podium = FINAL_MODEL.predict_proba(X_test)[:, 1] >= FINAL_THRESHOLD
false_alarm = said_podium & (y_test.to_numpy() == 0)
print(f"test false alarms: {false_alarm.sum()}; did not finish the race: {(~finished[false_alarm]).sum()} "
      f"({(~finished[false_alarm]).mean():.0%}), compared with {(~finished).mean():.0%} of all test starters")

# 4) feature drift (no labels): how far does each feature's test mean move, in training standard deviations?
drift = pd.DataFrame({"train mean": X_train[MODEL_FEATURES].mean(), "test mean": X_test[MODEL_FEATURES].mean(),
                      "test rows above the training maximum": (X_test[MODEL_FEATURES] > X_train[MODEL_FEATURES].max()).mean()})
drift["shift (training SDs)"] = (drift["test mean"] - drift["train mean"]) / X_train[MODEL_FEATURES].std()
display(drift.sort_values("shift (training SDs)", key=abs, ascending=False).head(6).round(3))

**What we see.**
* **Probabilities are realistic on train and validation, slightly high on test:** about 3.04 and 3.08 predicted podiums per race on train and validation, but 3.32 on the test years (average probability 0.163 against a real podium rate of 0.151). The ranking is not hurt (test PR-AUC 0.702), but about 0.3 extra "podium" probability per race means a fixed threshold says "yes" a little too often.
* **The drift check explains why:** the points features moved most. Before a race in 2022-2024 a driver has on average 58.8 points (36.4 in the training years) and a team 118 (73): seasons now have 22-24 races and sprint races, so there are more points to collect. The model learned "more points = more likely podium", so every test driver looks a little stronger. Pit-lane starts are also more common (2.7% of test starts vs 0.3%, engine-penalty starts), and qualifying laps are faster. Less than 1% of test rows lie above the training maximum of any single feature, so this is a shift, not values the model has never seen.
* **Era sensitivity, per season:** test PR-AUC 0.699 in 2022, 0.754 in 2023 and 0.643 in 2024. 2023 was dominated by one team and is the easiest to predict; 2024 had four different winning teams and is the hardest. How competitive a season is matters at least as much as the technical era.
* **False alarms:** 84 on the test set; 25% of those cars did not finish the race, compared with 14% of all test starters. Some false alarms are cars that broke down or crashed, something no pre-race model can see.

**Limitations of the models**

| Model | Main limitation in this project |
|---|---|
| Logistic regression (final) | Linear: no interactions unless they are built as features; the weights of correlated features (grid, qualifying, standings, team circuit counts) are not individually interpretable; it extrapolates linearly, so drifting features (points) shift its probabilities. |
| Random forest | Overfits with small leaves (train PR-AUC 0.93 with `min_samples_leaf=5`); probabilities are vote shares, not calibrated; cannot extrapolate beyond training values. |
| Gradient boosting | Sensitive to tree size and number of trees (default trees: train 0.97 vs validation 0.73; 800 small trees: train 0.86 vs validation 0.72); many hyperparameters for a small validation set. |
| Shallow neural network | Stopping epoch chosen on validation, so its validation score is optimistic (0.754 on validation, 0.688 on test); wider or deeper networks did not help; a black box without XAI. |
| k-NN, Naive Bayes (explored) | k-NN weighs every feature equally; Naive Bayes assumes independent features, which our correlated features break (validation PR-AUC 0.677 and 0.292). |

**Limitations of the data and the method**
* **Era sensitivity:** rules, cars and the points system change. The test years are a new technical era (2022 ground-effect cars) that the model has never seen; `era_ordinal` cannot be a model input because its test value never occurs in training (4.2). The small validation-to-test drop shows the relative features (grid, qualifying, championship position) still transfer, but the absolute points features drift (check 4).
* **Points drift:** `pts_before` and `cons_pts_before` count points, and the number of points available per season has grown. A version relative to the leader (or the championship position alone, which is already scale-free) would be more stable across eras; we leave this to the feature owner.
* **Qualifying coverage:** the table starts in 2006, the first season of today's knockout qualifying, so 2003-2005 are not used.
* **Missing values carry meaning:** in the training years a missing qualifying time almost always means a driver who set no time and started near the back, so the model reads "missing" as a bad sign (4.13: a pole-sitter without qualifying data and championship position drops from 72% to 6%). For a record with a data gap the prediction is unreliable.
* **No flag for features that are never missing in training:** only the features with gaps in the training years get a missing-value flag (4.2). If another feature (for example `grid_fixed`) is missing in a new record, it is silently replaced by the training median. These features are always known at the prediction cutoff, so a record without them should be treated as invalid input.
* **Raw lap time:** `best_qual_s` depends on the circuit length, so it is only comparable within one race (`gap_to_pole_pct` is the comparable version).
* **Small validation set:** 117 podiums; the bootstrap standard error of PR-AUC is about 0.04, larger than every difference between the tuned models, so the ranking of the models is uncertain.
* **The threshold is chosen for F1**, which counts a missed podium and a false alarm as equally bad. Another use (for example a betting or broadcast decision) weighs them differently and needs a different threshold.
* **The task itself:** incidents, penalties, safety cars, weather and strategy happen during the race and are invisible before it (Hamilton, Italy 2020, in 4.11). No pre-race model can predict them.

### 4.13 Hand-off to the inference function (P1)

P1's inference function uses the objects below. `FINAL_MODEL` is one pipeline (column selection + preprocessing + model), so it takes the columns in P3's `FEATURES` (it ignores `era_ordinal` itself) and handles missing values itself (4.2). Usage:

```python
proba = FINAL_MODEL.predict_proba(record[FEATURES].astype(float))[:, 1]   # podium probability
podium = proba >= FINAL_THRESHOLD                                         # yes / no decision
```

The check below runs the final model on one complete validation record and on the same record with its qualifying and championship values removed (as for a driver without a qualifying time in the first race of a season).

In [ ]:
complete_record = X_val[X_val[MODEL_FEATURES].notna().all(axis=1)].iloc[[0]]
incomplete_record = complete_record.copy()
incomplete_record[["qual_position", "best_qual_s", "gap_to_pole_pct", "pos_before", "cons_pos_before"]] = np.nan
check = pd.concat([complete_record, incomplete_record]).assign(record=["complete", "incomplete"]).set_index("record")
check["podium probability"] = FINAL_MODEL.predict_proba(check[FEATURES])[:, 1].round(3)
check["podium (threshold)"] = check["podium probability"] >= FINAL_THRESHOLD
assert check["podium probability"].notna().all()
display(check[["grid_fixed", "qual_position", "pts_before", "pos_before", "podium probability", "podium (threshold)"]])
print(f"FINAL_NAME = {FINAL_NAME!r}, FINAL_THRESHOLD = {FINAL_THRESHOLD}, "
      f"model input = {len(FEATURES)} columns of FEATURES, of which {len(MODEL_FEATURES)} are used")